In [ ]:
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

if tf.config.list_physical_devices('GPU'):
    print('✅ GPU is active — training will be fast (~10 mins)')
else:
    print('⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU')

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ GPU is active — training will be fast (~10 mins)


ValueError: mount failed

In [ ]:
import os, zipfile, gdown

# ── Your Google Drive file ID (extracted from your share link) ──
FILE_ID     = '1SKx-CryE40-v5KSUgIZYv3lQe35Wa_bS'
ZIP_PATH    = '/content/plantvillage.zip'
EXTRACT_DIR = '/content/dataset'

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Step 1: Download from Google Drive using the file ID
print('⬇️  Downloading dataset from Google Drive...')
gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', ZIP_PATH, quiet=False)

# Step 2: Extract the zip
if os.path.exists(ZIP_PATH):
    print('\n📦 Extracting dataset... (takes ~2 mins)')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('✅ Dataset extracted!')
else:
    print('❌ Download failed. Try sharing the file as "Anyone with the link".')

# Step 3: Show what was extracted
folders = [f for f in os.listdir(EXTRACT_DIR) if os.path.isdir(os.path.join(EXTRACT_DIR, f))]
print(f'\nFound {len(folders)} class folders')
print('First 5:', folders[:5])

In [ ]:
import os

def find_dataset_root(base_path):
    """
    Walk through extracted folders to find the directory
    that contains the 38 class subfolders.
    """
    for root, dirs, files in os.walk(base_path):
        # If this folder has 30+ subfolders, it's likely our dataset root
        if len(dirs) >= 30:
            print(f'✅ Dataset root found: {root}')
            print(f'   Contains {len(dirs)} class folders')
            return root
    return base_path

DATA_DIR = find_dataset_root('/content/dataset')
print(f'\nUsing DATA_DIR = "{DATA_DIR}"')

# Count total images
total = sum(len(files) for _, _, files in os.walk(DATA_DIR))
print(f'Total images found: {total:,}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Configuration
IMG_SIZE    = 224
BATCH_SIZE  = 64      # changed from 32 → 64 (faster)
EPOCHS_P1   = 5       # changed from 10 → 5 (faster)
EPOCHS_P2   = 3       # changed from 5 → 3 (faster)
MODEL_SAVE  = '/content/drive/MyDrive/PlantVillage/plant_disease_model.h5'

print('✅ Libraries imported successfully')
print(f'   Image size  : {IMG_SIZE}x{IMG_SIZE}')
print(f'   Batch size  : {BATCH_SIZE}')
print(f'   Total epochs: {EPOCHS_P1 + EPOCHS_P2}')

In [ ]:
import random
from PIL import Image

# Get all class folders
class_folders = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

print(f'Total classes: {len(class_folders)}')

# Plot 12 random sample images
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Sample images from PlantVillage dataset', fontsize=14, fontweight='bold')

sample_classes = random.sample(class_folders, min(12, len(class_folders)))

for ax, cls in zip(axes.flatten(), sample_classes):
    cls_path = os.path.join(DATA_DIR, cls)
    img_files = os.listdir(cls_path)
    if img_files:
        img_path = os.path.join(cls_path, random.choice(img_files))
        img = Image.open(img_path).resize((224, 224))
        ax.imshow(img)
        # Format label: Tomato___Early_blight → Tomato\nEarly blight
        parts = cls.split('___')
        label = f"{parts[0]}\n{parts[1].replace('_',' ')}" if len(parts)>1 else cls
        color = 'green' if 'healthy' in cls.lower() else 'red'
        ax.set_title(label, fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Green title = healthy | Red title = diseased')

In [ ]:
# Training: augment images to make model more robust
train_datagen = ImageDataGenerator(
    rescale           = 1.0/255,   # normalize pixels [0,255] → [0,1]
    rotation_range    = 30,        # rotate up to 30 degrees
    width_shift_range = 0.2,       # shift horizontally
    height_shift_range= 0.2,       # shift vertically
    shear_range       = 0.2,       # slant transformation
    zoom_range        = 0.2,       # random zoom
    horizontal_flip   = True,      # mirror left-right
    fill_mode         = 'nearest', # fill empty pixels
    validation_split  = 0.2        # 80% train, 20% validation
)

# Validation: only rescale, NO augmentation
val_datagen = ImageDataGenerator(
    rescale          = 1.0/255,
    validation_split = 0.2
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    subset      = 'training',
    shuffle     = True
)

val_gen = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    subset      = 'validation',
    shuffle     = False
)

NUM_CLASSES = len(train_gen.class_indices)
CLASS_NAMES = list(train_gen.class_indices.keys())

print(f'✅ Training samples   : {train_gen.samples:,}')
print(f'✅ Validation samples : {val_gen.samples:,}')
print(f'✅ Number of classes  : {NUM_CLASSES}')

In [ ]:
# Load MobileNetV2 without its top classification layer
base_model = MobileNetV2(
    input_shape = (IMG_SIZE, IMG_SIZE, 3),
    include_top = False,      # remove the original 1000-class head
    weights     = 'imagenet'  # use pretrained weights
)
base_model.trainable = False  # freeze: don't update base weights yet

# Build our custom model on top
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),         # (7,7,1280) → (1280,)
    layers.Dense(256, activation='relu'),    # learn plant-specific features
    layers.Dropout(0.3),                     # prevent overfitting
    layers.Dense(NUM_CLASSES, activation='softmax')  # output probabilities
], name='PlantDiseaseDetector')

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

# Print summary
model.summary()
print(f'\nTotal parameters    : {model.count_params():,}')
print(f'Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

In [ ]:
import tensorflow as tf

callbacks = [
    # Stop if validation loss doesn't improve for 3 epochs
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True, verbose=1
    ),
    # Save best model to Google Drive automatically
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE, monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    # Reduce learning rate if stuck
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, verbose=1
    )
]

print('🚀 Phase 1: Training top layers...')
print(f'   Epochs: {EPOCHS_P1} | Learning rate: 1e-3')
print(f'   Base model: FROZEN\n')

history1 = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs          = EPOCHS_P1,
    callbacks       = callbacks
)

p1_acc = max(history1.history['val_accuracy']) * 100
print(f'\n✅ Phase 1 complete! Best val accuracy: {p1_acc:.2f}%')

In [ ]:
# Unfreeze the last 30 layers of base model
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with much smaller learning rate
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

trainable_count = sum([tf.size(w).numpy() for w in model.trainable_weights])
print('🔬 Phase 2: Fine-tuning...')
print(f'   Epochs: {EPOCHS_P2} | Learning rate: 1e-5')
print(f'   Last 30 base layers: UNFROZEN')
print(f'   Trainable parameters: {trainable_count:,}\n')

history2 = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs          = EPOCHS_P2,
    callbacks       = callbacks
)

p2_acc = max(history2.history['val_accuracy']) * 100
print(f'\n✅ Phase 2 complete! Best val accuracy: {p2_acc:.2f}%')
print(f'💾 Model saved to Google Drive: {MODEL_SAVE}')

In [ ]:
# Combine both phases
acc     = history1.history['accuracy']     + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss    = history1.history['loss']         + history2.history['loss']
val_loss= history1.history['val_loss']     + history2.history['val_loss']
epochs  = range(1, len(acc) + 1)
p1_end  = len(history1.history['accuracy'])  # mark where phase 2 begins

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Results — Plant Disease Detection', fontsize=14, fontweight='bold')

for ax, train_vals, val_vals, title, ylabel in zip(
    axes,
    [acc, loss],
    [val_acc, val_loss],
    ['Model Accuracy', 'Model Loss'],
    ['Accuracy', 'Loss']
):
    ax.plot(epochs, train_vals, 'b-o', label='Training',   markersize=4, linewidth=2)
    ax.plot(epochs, val_vals,   'r-o', label='Validation', markersize=4, linewidth=2)
    ax.axvline(x=p1_end+0.5, color='gray', linestyle='--', alpha=0.7, label='Fine-tune starts')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/PlantVillage/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved to Google Drive: training_curves.png')

In [ ]:
print('🔍 Running predictions on validation set...')

# Get predictions (shuffle=False is critical — keeps label order)
val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = val_gen.classes

# Overall accuracy
accuracy = np.mean(y_pred == y_true) * 100
print(f'\n✅ Overall Validation Accuracy: {accuracy:.2f}%')

# Shortened class names for display
short_names = [c.split('___')[-1].replace('_',' ')[:25] for c in CLASS_NAMES]

# Full classification report
print('\n📋 Classification Report (precision, recall, F1 per class):')
print(classification_report(y_true, y_pred, target_names=short_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(18, 15))
sns.heatmap(
    cm,
    annot    = True,
    fmt      = 'd',
    cmap     = 'Blues',
    xticklabels = short_names,
    yticklabels = short_names
)
plt.title('Confusion Matrix — Plant Disease Detection', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/PlantVillage/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved to Google Drive: confusion_matrix.png')

In [ ]:
from google.colab import files
from PIL import Image
import io

print('📷 Upload a leaf image to test the model...')
uploaded = files.upload()

for filename, data in uploaded.items():
    # Load and preprocess image
    img        = Image.open(io.BytesIO(data)).convert('RGB')
    img_resized= img.resize((IMG_SIZE, IMG_SIZE))
    img_array  = np.array(img_resized, dtype=np.float32) / 255.0
    img_array  = np.expand_dims(img_array, axis=0)   # shape: (1, 224, 224, 3)

    # Predict
    preds      = model.predict(img_array, verbose=0)[0]
    top3_idx   = np.argsort(preds)[::-1][:3]

    # Display results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Show image
    ax1.imshow(img)
    ax1.axis('off')
    best_label = CLASS_NAMES[top3_idx[0]]
    plant, disease = best_label.split('___') if '___' in best_label else (best_label, 'Unknown')
    color = 'green' if 'healthy' in best_label.lower() else 'red'
    ax1.set_title(f'{plant}\n{disease.replace("_"," ")}', fontsize=13, fontweight='bold', color=color)

    # Bar chart of top-3
    labels  = [CLASS_NAMES[i].split('___')[-1].replace('_',' ')[:30] for i in top3_idx]
    scores  = [preds[i]*100 for i in top3_idx]
    colors  = ['#2ecc71' if 'healthy' in CLASS_NAMES[i].lower() else '#e74c3c' for i in top3_idx]
    bars    = ax2.barh(labels[::-1], scores[::-1], color=colors[::-1])
    ax2.set_xlabel('Confidence (%)', fontsize=11)
    ax2.set_title('Top 3 Predictions', fontsize=12, fontweight='bold')
    ax2.set_xlim(0, 105)
    for bar, score in zip(bars, scores[::-1]):
        ax2.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
                 f'{score:.1f}%', va='center', fontsize=11, fontweight='bold')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/PlantVillage/prediction_result.png', dpi=150)
    plt.show()

    print(f'\n🌿 Predicted : {best_label.replace("_"," ")}')
    print(f'📊 Confidence: {preds[top3_idx[0]]*100:.2f}%')

In [ ]:
final_loss, final_acc = model.evaluate(val_gen, verbose=0)

print('=' * 55)
print('         PLANT DISEASE DETECTION — RESULTS')
print('=' * 55)
print(f'  Dataset          : PlantVillage')
print(f'  Total images     : {train_gen.samples + val_gen.samples:,}')
print(f'  Classes          : {NUM_CLASSES}')
print(f'  Base model       : MobileNetV2 (ImageNet pretrained)')
print(f'  Training images  : {train_gen.samples:,}')
print(f'  Validation images: {val_gen.samples:,}')
print(f'  Total epochs     : {EPOCHS_P1 + EPOCHS_P2}')
print('-' * 55)
print(f'  Final Val Accuracy : {final_acc*100:.2f}%')
print(f'  Final Val Loss     : {final_loss:.4f}')
print('=' * 55)
print('\n📁 Files saved to Google Drive (MyDrive/PlantVillage/):')
print('   plant_disease_model.h5   ← trained model')
print('   training_curves.png      ← for your report')
print('   confusion_matrix.png     ← for your report')
print('   prediction_result.png    ← demo result')
print('\n📝 For your report, include:')
print('   1. Dataset description (54k images, 38 classes)')
print('   2. Model architecture diagram (MobileNetV2 + head)')
print('   3. Training curves graph')
print('   4. Confusion matrix')
print('   5. Classification report (precision/recall/F1)')
print('   6. Sample prediction screenshots')